## Setup

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision.transforms import v2
from torchvision.io import decode_image

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
bmt_dir = "/content/drive/MyDrive/BMT_embeddings"

Mounted at /content/drive


## Dataset (BMT)

In [ ]:
class BMT(Dataset):
    def __init__(self, label, img_dir="/content/drive/MyDrive/BMT", transform=None, target_transform=None):
        '''
        label:
          0 <-> 'NIL'
          1 <-> 'LSIL'
          2 <-> 'HSIL'
        '''
        self.label_dict = {'NIL':0, 'LSIL':1, 'HSIL':2}
        self.label = label
        self.img_dir = img_dir + "/" + self.label
        self.size = self.count_files()
        self.transform = transform
        self.target_transform = target_transform

    def count_files(self):
      total_arquivos = 0
      for raiz, diretorios, arquivos in os.walk(self.img_dir):
          total_arquivos += len(arquivos)

      return total_arquivos

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        img_path = self.img_dir + "/" + f'{idx+1}.png'
        image = decode_image(img_path)
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(self.label)
        return image, self.label_dict[self.label]

In [ ]:
transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])

nil_data = BMT(label='NIL', transform=transform)
lsil_data = BMT(label='LSIL', transform=transform)
hsil_data = BMT(label='HSIL', transform=transform)

## Dataset (BMT embeddings)

In [ ]:
class BMT_embeddings(Dataset):
    def __init__(self, label, img_dir=bmt_dir):
        '''
        label:
          0 <-> 'NIL'
          1 <-> 'LSIL'
          2 <-> 'HSIL'
        '''
        self.label_dict = {'NIL':0, 'LSIL':1, 'HSIL':2}
        self.label = label
        self.img_dir = img_dir + "/" + self.label

    def __len__(self):
        return 200

    def __getitem__(self, idx):
        embed_path = self.img_dir + "/" + f'{idx+1}.pt'
        embed = torch.load(embed_path, map_location="cpu")
        return embed, self.label_dict[self.label]

In [ ]:
nil_embed_data = BMT_embeddings(label='NIL')
lsil_embed_data = BMT_embeddings(label='LSIL')
hsil_embed_data = BMT_embeddings(label='HSIL')

## Dataloader (BMT embeddings)

In [ ]:
nil_dataloader = DataLoader(nil_embed_data, batch_size=1)
lsil_dataloader = DataLoader(lsil_embed_data, batch_size=1)
hsil_dataloader = DataLoader(hsil_embed_data, batch_size=1)

## Architecture (ABMIL)

In [ ]:
'''
Feito com auxílio do  Claude Code (Opus 5.5 / High)
'''

import torch.nn as nn

class GatedAttentionMIL(nn.Module):
    """ABMIL com atenção com porta (Ilse et al., 2018)."""

    def __init__(self, in_dim=1024, hid=256, att_hid=128, n_classes=3, p_drop=0.25):
        super().__init__()
        # projeção: reduz 1024 → 256 antes da atenção
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hid),
            nn.ReLU(),
            nn.Dropout(p_drop),
        )
        # dois ramos da atenção com porta
        self.att_V = nn.Sequential(nn.Linear(hid, att_hid), nn.Tanh())
        self.att_U = nn.Sequential(nn.Linear(hid, att_hid), nn.Sigmoid())
        self.att_w = nn.Linear(att_hid, 1)

        self.classifier = nn.Linear(hid, n_classes)

    def forward(self, bags):                    # bags: [B, N, 1024]
        h = self.proj(bags)                     # [B, N, 256]

        a = self.att_w(self.att_V(h) * self.att_U(h))   # [B, N, 1]
        a = torch.softmax(a, dim=1)                     # pesos somam 1 no bag

        return a.argmax().item()                        # estamos preocupados apenas em mapear a atenção agora

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
abmil = GatedAttentionMIL()
state = torch.load("/content/drive/MyDrive/abmil.pth", map_location='cpu')
abmil.load_state_dict(state)
abmil.to(device)

GatedAttentionMIL(
  (proj): Sequential(
    (0): Linear(in_features=1024, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.25, inplace=False)
  )
  (att_V): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): Tanh()
  )
  (att_U): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): Sigmoid()
  )
  (att_w): Linear(in_features=128, out_features=1, bias=True)
  (classifier): Linear(in_features=256, out_features=3, bias=True)
)

## Mapping attention

In [ ]:
nil_attention = []
lsil_attention = []
hsil_attention = []

In [ ]:
abmil.eval()
with torch.no_grad():
  for x, _ in nil_dataloader:
    x = x.to(device)
    a = abmil(x)
    nil_attention.append(a)
  for x, _ in lsil_dataloader:
    x = x.to(device)
    a = abmil(x)
    lsil_attention.append(a)
  for x, _ in hsil_dataloader:
    x = x.to(device)
    a = abmil(x)
    hsil_attention.append(a)

In [ ]:
torch.save(nil_attention, "/content/drive/MyDrive/nil_attention.pt")
torch.save(lsil_attention, "/content/drive/MyDrive/lsil_attention.pt")
torch.save(hsil_attention, "/content/drive/MyDrive/hsil_attention.pt")

In [ ]:
nil_attention = torch.load("/content/drive/MyDrive/nil_attention.pt")
lsil_attention = torch.load("/content/drive/MyDrive/lsil_attention.pt")
hsil_attention = torch.load("/content/drive/MyDrive/hsil_attention.pt")

## Tile extraction (highest attention scores)

In [ ]:
def tile_extraction(tensor, idx):
    height, width = tensor.shape[1], tensor.shape[2]
    count = 0
    for y in range(0, height, 224):
        for x in range(0, width, 224):
            if count != idx:
              count += 1
              continue

            y_start = min(y, height - 224)
            x_start = min(x, width - 224)

            patch = tensor[:, y_start:y_start+224, x_start:x_start+224]
            return patch
    return

In [ ]:
nil_tiles_attention = []
lsil_tiles_attention = []
hsil_tiles_attention = []

In [ ]:
from torchvision.utils import save_image

for i, idx in enumerate(nil_attention):
  tile_attention = tile_extraction(nil_data[i][0], idx)
  #save_image(tile_attention, f"/content/drive/MyDrive/BMT_tiles_attention/NIL/{i+1}.png")
  nil_tiles_attention.append(nil_embed_data[i][0][idx])

for i, idx in enumerate(lsil_attention):
  tile_attention = tile_extraction(lsil_data[i][0], idx)
  #save_image(tile_attention, f"/content/drive/MyDrive/BMT_tiles_attention/LSIL/{i+1}.png")
  lsil_tiles_attention.append(lsil_embed_data[i][0][idx])

for i, idx in enumerate(hsil_attention):
  tile_attention = tile_extraction(hsil_data[i][0], idx)
  #save_image(tile_attention, f"/content/drive/MyDrive/BMT_tiles_attention/HSIL/{i+1}.png")
  hsil_tiles_attention.append(hsil_embed_data[i][0][idx])

In [ ]:
torch.save(nil_tiles_attention, "/content/drive/MyDrive/nil_tiles_attention.pt")
torch.save(lsil_tiles_attention, "/content/drive/MyDrive/lsil_tiles_attention.pt")
torch.save(hsil_tiles_attention, "/content/drive/MyDrive/hsil_tiles_attention.pt")